In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# imports
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# get all csv files in the data directory for run of interest
data_dir = Path("output/runs/1/data")
csv_files = list(data_dir.glob("*.csv"))
print(f"Found {len(csv_files)} CSV files.")

# read all files with mixed types
arrays = []
for i, f in enumerate(csv_files):
    arr = np.genfromtxt(f, delimiter=',', dtype=None, encoding='utf-8')
    arrays.append(arr)
    if (i + 1) % 1000 == 0:
        print(f"Loaded {i+1}/{len(csv_files)} files")

# stack all arrays vertically
data = np.concatenate(arrays)
# convert data to pandas df
df = pd.DataFrame(data)
# remove all rows that are exact duplicates, since they are removing the rows of column names
df = df.drop_duplicates()
# make first row the header
df.columns = df.iloc[0]
df = df[1:].reset_index(drop=True)

In [ ]:
df.head()

In [ ]:
# get distribution of accuracy via histogram
plt.figure(figsize=(10, 6))
plt.hist(df['test_acc'].astype(float) * 100, bins=10, edgecolor='black', alpha=0.7)
plt.title('Test Accuracy Across the Multiverse', fontsize=14, fontweight='bold')
plt.xlabel('Test Accuracy (%)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.gca().xaxis.set_major_formatter(mtick.PercentFormatter())
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Comprehensive Variability Analysis: All Decision Dimensions

We analyze the inter-bucket and intra-bucket variability for each decision in the pipeline.

In [ ]:
# define all decision dimensions
decision_dims = [
    ("mv_dim_missing_strategy", "Missing Strategy"),
    ("mv_dim_model", "Model"),
    ("mv_dim_new_job_no_notif_threshold", "New Job No-Notification Threshold"),
    ("mv_dim_new_job_notif_threshold", "New Job Notification Threshold"),
    ("mv_dim_spell_gap_threshold", "Spell Gap Threshold"),
    ("mv_dim_split_seed", "Train-Test Split Seed"),
]

In [ ]:
# boxplots for each decision dimension
fig, axes = plt.subplots(2, 3, figsize=(14, 10))
axes = axes.flatten()

for i, (dim_col, dim_name) in enumerate(decision_dims):
    ax = axes[i]
    
    # Get unique values and sort them
    unique_vals = sorted(df[dim_col].unique(), key=lambda x: str(x))
    
    # Create boxplot data - cast to float since values are strings; convert to percent
    box_data = [df[df[dim_col] == val]['test_acc'].astype(float).values * 100 for val in unique_vals]
    
    bp = ax.boxplot(box_data, patch_artist=True)
    
    # Color the boxes
    colors = plt.cm.Set3(np.linspace(0, 1, len(unique_vals)))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
    
    ax.set_xticklabels([str(v)[:15] for v in unique_vals], rotation=45, ha='right', fontsize=10)
    ax.set_ylabel('Test Accuracy (%)', fontsize=12)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=100, decimals=0))
    ax.tick_params(axis='y', labelsize=11)
    ax.tick_params(axis='x', labelsize=10)
    ax.set_title(f'{dim_name}', fontsize=11, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Test Accuracy Distribution by Each Decision Dimension', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()